# Insert and Load Capacity

This notebook demonstrates how to insert vectors while checking index capacity, defer loading with `load=False`, and then explicitly load the index before search. If the target index is not loadable, the notebook unloads currently loaded indexes before loading the new target index.


## Import SDK

Import the `pyenvector` package to use the enVector Python APIs in the following cells.

For installation details, see [SDK installation](https://docs.envector.io/get-started/installation/client-sdk) in [enVector Documents](https://docs.envector.io/).


In [ ]:
import pyenvector as ev

## Insert and Load Workflow

### 1. Initialize

This cell initializes the SDK with:

1. `address`: the enVector server endpoint.
2. `key_path` and `key_id`: the local key bundle used for encrypted index operations.
3. `index_name`: the base name used when a new index must be created.

For more details, see [Initialize](https://docs.envector.io/user-guide/initialize) in [enVector Documents](https://docs.envector.io/).


In [ ]:
import os
address = os.getenv("ENVECTOR_ADDRESS", "localhost:50050")

key_path = "./keys"
key_id = "insert-load"
index_name = "insert_load_index"
ev.init(
    address=address,
    # access_token="...", # if needed
    key_path=key_path,
    key_id=key_id,
)

### 2. Select or Create an Index

The next cell reads the current index list.

- If no index exists, it creates a new index named `{index_name}_1` with `dim=512`.
- If an index already exists, it reuses the most recently listed index.

The selected index name is stored and reused in the later insert, load, search, and cleanup steps.


In [ ]:
index_list = ev.get_index_list()
if len(index_list) == 0:
    current_index_name = f"{index_name}_{len(index_list)+1}"
    index = ev.create_index(current_index_name, dim=512)
else:
    current_index_name = index_list[-1]
    index = ev.Index(current_index_name)


### 3. Check Capacity and Insert Vectors

This step checks whether the selected index still has enough insert capacity for `num_vectors = 100`. The `remaining_insertable_vectors` value reports how many additional vectors can be inserted into the index before its insert capacity is exhausted.

- If the current index does not have enough remaining capacity, the code creates a new `512`-dimensional index and switches to that index.
- It generates `100` random vectors, normalizes them, and creates simple string metadata.
- It inserts the vectors with `load=False`, so the insert step does not automatically load the index into memory.

This notebook therefore separates insertion from loading on purpose.


In [ ]:
# check index insertable
available_num_vectors = index.remaining_insertable_vectors
num_vectors = 100
if num_vectors > available_num_vectors:
    current_index_name = f"{index_name}_{len(ev.get_index_list())+1}"
    index = ev.create_index(current_index_name, dim=512)

import numpy as np

vecs = np.random.rand(num_vectors, 512)
vecs = vecs / np.linalg.norm(vecs, axis=1, keepdims=True)  # normalize for IP
metadata = [f"Item {i+1}" for i in range(num_vectors)]

index.insert(vecs, metadata, load=False)


### 4. Explicitly Load and Search

Because the insert step used `load=False`, the index is loaded explicitly before search.

- The code opens the same index selected earlier.
- If that index is not currently loadable, it first unloads other indexes so the target index can be loaded.
- The `loadable` value is an estimate based on the available idle memory space.
- It then loads the target index and runs a top-3 search with the first inserted vector.
- The search request asks for the `metadata` field in the results.

For more details, see [Search](https://docs.envector.io/user-guide/search) in [enVector Documents](https://docs.envector.io/).


In [ ]:
search_index = ev.Index(current_index_name)
if not search_index.loadable:
    for i in ev.get_index_list():
        unload_index = ev.Index(i)
        unload_index.unload()

search_index.load()

query = vecs[0]
results = search_index.search(query, top_k=3, output_fields=["metadata"])[0]
print(results)


### Clean Up

The final cells remove the index used in this notebook and delete the registered key.


In [ ]:
ev.drop_index(current_index_name)


In [ ]:
ev.delete_key(key_id)